In [1]:
import polars as pl
from depersonal_lib.core.utils import MLPerformanceEvaluator
from depersonal_lib.core.utils import ShowStatistic

## Пример использования ShowStatistic

In [2]:
# Создаем тестовые данные
original_data = {
    "user_id": [101, 102, 103, 104, 105, 106],
    "age": [23, 31, 48, 29, 31, 52],
    "income": [50000.0, 75000.0, 120000.0, 60000.0, None, 150000.0],
    "city": ["New York", "London", "Tokyo", "London", "Paris", "Tokyo"]
}
original_df = pl.DataFrame(original_data)
original_df

user_id,age,income,city
i64,i64,f64,str
101,23,50000.0,"""New York"""
102,31,75000.0,"""London"""
103,48,120000.0,"""Tokyo"""
104,29,60000.0,"""London"""
105,31,null,"""Paris"""
106,52,150000.0,"""Tokyo"""


In [3]:
anonymized_data = {
    # удалили id
    "age": ['1-30', '31-60','31-60','1-30','31-60','31-60'], # Обобщили до групп возраст
    "income": [50000.0, 75000.0, 120000.0, 60000.0, 91000.0, 150000.0], # Заменили None на среднее
    "city": ["USA", "Europe", "Asia", "Europe", "Europe", "Asia"] # Укрупнили категории
}
anonymized_df = pl.DataFrame(anonymized_data)
anonymized_df

age,income,city
str,f64,str
"""1-30""",50000.0,"""USA"""
"""31-60""",75000.0,"""Europe"""
"""31-60""",120000.0,"""Asia"""
"""1-30""",60000.0,"""Europe"""
"""31-60""",91000.0,"""Europe"""
"""31-60""",150000.0,"""Asia"""


In [4]:
# 3. Используем класс ShowStatistic для сравнения
comparator = ShowStatistic(original_df, anonymized_df)
comparison_results = comparator.compare_statistics()

In [5]:
print("\n--- Результаты сравнения статистики ---")
print(comparison_results)


--- Результаты сравнения статистики ---
shape: (4, 9)
┌─────────┬───────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ column  ┆ orig_mean ┆ anon_mean  ┆ orig_std  ┆ … ┆ orig_miss ┆ anon_miss ┆ orig_uniq ┆ anon_uniq │
│ ---     ┆ ---       ┆ ---        ┆ ---       ┆   ┆ ing_rate  ┆ ing_rate  ┆ ue_count  ┆ ue_count  │
│ str     ┆ f64       ┆ str        ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│         ┆           ┆            ┆           ┆   ┆ f64       ┆ str       ┆ i64       ┆ str       │
╞═════════╪═══════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ age     ┆ 35.666667 ┆ null       ┆ 11.552777 ┆ … ┆ 0.0       ┆ 0         ┆ 5         ┆ 2         │
│ city    ┆ null      ┆ null       ┆ null      ┆ … ┆ 0.0       ┆ 0         ┆ 4         ┆ 3         │
│ income  ┆ 91000.0   ┆ 91000      ┆ 42485.291 ┆ … ┆ 0.166667  ┆ 0         ┆ 6         ┆ 6         │
│         ┆           ┆            ┆

In [6]:
comparison_results.write_excel('comparison.xlsx')

## Пример использования MLPerformanceEvaluator

In [2]:
# Загрузка данных
df = pl.read_csv("adult.csv")
print("Исходные данные:")
df.head()

Исходные данные:


age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
25,"""Private""",226802,"""11th""",7,"""Never-married""","""Machine-op-inspct""","""Own-child""","""Black""","""Male""",0,0,40,"""United-States""","""<=50K"""
38,"""Private""",89814,"""HS-grad""",9,"""Married-civ-spouse""","""Farming-fishing""","""Husband""","""White""","""Male""",0,0,50,"""United-States""","""<=50K"""
28,"""Local-gov""",336951,"""Assoc-acdm""",12,"""Married-civ-spouse""","""Protective-serv""","""Husband""","""White""","""Male""",0,0,40,"""United-States""",""">50K"""
44,"""Private""",160323,"""Some-college""",10,"""Married-civ-spouse""","""Machine-op-inspct""","""Husband""","""Black""","""Male""",7688,0,40,"""United-States""",""">50K"""
18,"""?""",103497,"""Some-college""",10,"""Never-married""","""?""","""Own-child""","""White""","""Female""",0,0,30,"""United-States""","""<=50K"""


In [3]:
original_df = df.clone()
anonymized_df = df.drop('education')

In [4]:
# Инициализация оценщика
evaluator = MLPerformanceEvaluator(
    original_df=original_df,
    anonymized_df=anonymized_df,
    target_column="income"  # Целевая переменная
)

In [5]:
# Оценка производительности ML
results = evaluator.evaluate_ml_performance(model_type="auto")

[('num', StandardScaler(), ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']), ('cat', OneHotEncoder(handle_unknown='ignore'), ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country'])]
[('num', StandardScaler(), ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']), ('cat', OneHotEncoder(handle_unknown='ignore'), ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country'])]
[0 0 1 ... 1 0 1]
[0 0 1 ... 1 0 1]


In [6]:
# Вывод результатов
print(f"Тип задачи: {results['model_type']}")
print("\nМетрики производительности:")
for metric, value in results["metrics"].items():
    if metric.endswith("_orig"):
        print(f"  Оригинальные данные ({metric.replace('_orig', '')}): {value:.4f}")
    elif metric.endswith("_anon"):
        print(f"  Обезличенные данные ({metric.replace('_anon', '')}): {value:.4f}")
    elif metric.endswith("_diff"):
        print(f"  Разница ({metric.replace('_diff', '')}): {value:.4f}")

Тип задачи: classification

Метрики производительности:
  Оригинальные данные (accuracy): 0.8570
  Обезличенные данные (accuracy): 0.8568
  Оригинальные данные (precision): 0.7364
  Обезличенные данные (precision): 0.7346
  Оригинальные данные (recall): 0.6074
  Обезличенные данные (recall): 0.6092
  Оригинальные данные (roc_auc): 0.9106
  Обезличенные данные (roc_auc): 0.9106
  Оригинальные данные (f1): 0.8520
  Обезличенные данные (f1): 0.8519
  Разница (accuracy): 0.0002
  Разница (precision): 0.0018
  Разница (recall): 0.0017
  Разница (roc_auc): 0.0000
  Разница (f1): 0.0001


In [7]:
# Вывод результатов
print(f"Тип задачи: {results['model_type']}")
print("\nМетрики производительности:")
for metric, value in results["metrics"].items():
    if metric.endswith("_orig"):
        print(f"  Оригинальные данные ({metric.replace('_orig', '')}): {value:.4f}")
    elif metric.endswith("_anon"):
        print(f"  Обезличенные данные ({metric.replace('_anon', '')}): {value:.4f}")
    elif metric.endswith("_diff"):
        print(f"  Разница ({metric.replace('_diff', '')}): {value:.4f}")

Тип задачи: classification

Метрики производительности:
  Оригинальные данные (accuracy): 0.8570
  Обезличенные данные (accuracy): 0.8570
  Оригинальные данные (precision): 0.7364
  Обезличенные данные (precision): 0.7364
  Оригинальные данные (recall): 0.6074
  Обезличенные данные (recall): 0.6074
  Оригинальные данные (roc_auc): 0.9106
  Обезличенные данные (roc_auc): 0.9106
  Оригинальные данные (f1): 0.8520
  Обезличенные данные (f1): 0.8520
  Разница (accuracy): 0.0000
  Разница (precision): 0.0000
  Разница (recall): 0.0000
  Разница (roc_auc): 0.0000
  Разница (f1): 0.0000


In [6]:
# Вывод результатов
print(f"Тип задачи: {results['model_type']}")
print("\nМетрики производительности:")
for metric, value in results["metrics"].items():
    if metric.endswith("_orig"):
        print(f"  Оригинальные данные ({metric.replace('_orig', '')}): {value:.4f}")
    elif metric.endswith("_anon"):
        print(f"  Обезличенные данные ({metric.replace('_anon', '')}): {value:.4f}")
    elif metric.endswith("_diff"):
        print(f"  Разница ({metric.replace('_diff', '')}): {value:.4f}")

Тип задачи: classification

Метрики производительности:
  Оригинальные данные (accuracy): 0.8570
  Обезличенные данные (accuracy): 0.8570
  Оригинальные данные (f1): 0.8520
  Обезличенные данные (f1): 0.8520
  Разница (accuracy): 0.0000
  Разница (f1): 0.0000
